# Cargas históricas de CSV a Firestore

Las celdas siguientes **escriben en Firestore**. No usar `Run All`: hay alternativas que repiten cargas. Elegir descarga de Kaggle o archivos locales y ejecutar solo las celdas correspondientes en un proyecto de pruebas propio.

Para explorar sin credenciales, usar `python scripts/inspeccionar_datos.py`. Ver [README.md](./README.md), [docs/ARQUITECTURA.md](./docs/ARQUITECTURA.md) y [SECURITY.md](./SECURITY.md).


In [ ]:
import os
import kagglehub
import pandas as pd
import firebase_admin
from firebase_admin import credentials, firestore

# Configurar Firebase con las credenciales
cred = credentials.ApplicationDefault()
try:
    firebase_admin.get_app()
except ValueError:
    firebase_admin.initialize_app(cred)
db = firestore.client()

# Descargar dataset desde Kaggle
path = kagglehub.dataset_download("wyattowalsh/basketball")
print("Dataset descargado en:", path)

# Ruta a la carpeta que contiene los CSV
csv_folder = f"{path}/csv"

# Iterar sobre todos los archivos CSV en la carpeta
for csv_file in os.listdir(csv_folder):
    if csv_file.endswith(".csv"):
        full_csv_path = os.path.join(csv_folder, csv_file)
        print(f"Subiendo archivo: {csv_file}")

        # Cargar el CSV en un DataFrame
        df = pd.read_csv(full_csv_path)

        # Crear una colección en Firestore con el nombre del archivo CSV (sin extensión)
        collection_name = os.path.splitext(csv_file)[0]  # Nombre de la colección
        collection_ref = db.collection(collection_name)

        # Subir los datos a Firestore
        for index, row in df.iterrows():
            doc_ref = collection_ref.document(str(index))  # Crear un documento con ID único
            doc_ref.set(row.to_dict())  # Convertir fila a diccionario y subir

        print(f"Datos de {csv_file} subidos correctamente a Firestore.")

print("Todos los archivos CSV se han subido correctamente a Firebase Firestore.")


In [ ]:
import os
import kagglehub
import pandas as pd
import firebase_admin
from firebase_admin import credentials, firestore

# Configurar Firebase con las credenciales
cred = credentials.ApplicationDefault()
try:
    firebase_admin.get_app()
except ValueError:
    firebase_admin.initialize_app(cred)
db = firestore.client()

# Descargar dataset desde Kaggle
path = kagglehub.dataset_download("wyattowalsh/basketball")
print("Dataset descargado en:", path)

# Ruta a la carpeta que contiene los CSV
csv_folder = f"{path}/csv"

# Lista de tablas (primera mitad)
tables = [
    'common_player_info.csv', 'draft_combine_stats.csv', 'draft_history.csv',
    'game.csv', 'game_info.csv', 'game_summary.csv', 'inactive_players.csv', 'line_score.csv'
]

# Subir la primera mitad de las tablas a Firestore
for csv_file in tables:
    full_csv_path = os.path.join(csv_folder, csv_file)
    print(f"Subiendo archivo: {csv_file}")

    # Cargar el CSV en un DataFrame
    df = pd.read_csv(full_csv_path)

    # Crear una colección en Firestore con el nombre del archivo CSV (sin extensión)
    collection_name = os.path.splitext(csv_file)[0]
    collection_ref = db.collection(collection_name)

    # Subir los datos a Firestore
    for index, row in df.iterrows():
        doc_ref = collection_ref.document(str(index))  
        doc_ref.set(row.to_dict())  

    print(f"Datos de {csv_file} subidos correctamente.")

print("Primera mitad de los archivos subidos a Firebase Firestore.")


In [ ]:
# Lista de tablas (segunda mitad)
tables = [
    'officials.csv', 'other_stats.csv', 'player.csv', 'play_by_play.csv',
    'team.csv', 'team_details.csv', 'team_history.csv', 'team_info_common.csv'
]

# Subir la segunda mitad de las tablas a Firestore
for csv_file in tables:
    full_csv_path = os.path.join(csv_folder, csv_file)
    print(f"Subiendo archivo: {csv_file}")

    # Cargar el CSV en un DataFrame
    df = pd.read_csv(full_csv_path)

    # Crear una colección en Firestore con el nombre del archivo CSV (sin extensión)
    collection_name = os.path.splitext(csv_file)[0]
    collection_ref = db.collection(collection_name)

    # Subir los datos a Firestore
    for index, row in df.iterrows():
        doc_ref = collection_ref.document(str(index))  
        doc_ref.set(row.to_dict())  

    print(f"Datos de {csv_file} subidos correctamente.")

print("Segunda mitad de los archivos subidos a Firebase Firestore.")


# Carga desde CSV locales — alternativa a Kaggle

Iniciar Jupyter desde la raíz del repositorio. La carpeta por defecto es `Dashboard/csv`; se puede definir `PI_CSV_DIR` antes de iniciar Jupyter. Ejecutar primero la celda que define `upload_to_firebase` y luego, si corresponde, su segunda lista. Estas listas son una selección de tablas, no todo el directorio.

`play_by_play.csv` no está incluido en main y no se solicita en esta alternativa local.


In [ ]:
import os
import pandas as pd
import firebase_admin
from firebase_admin import credentials, firestore

# Configurar Firebase con las credenciales
cred = credentials.ApplicationDefault()
try:
    firebase_admin.get_app()
except ValueError:
    firebase_admin.initialize_app(cred)
db = firestore.client()

# Ruta a la carpeta donde están los archivos CSV
csv_folder = os.environ.get("PI_CSV_DIR", os.path.join(os.getcwd(), "Dashboard", "csv"))

# Archivos CSV para subir en la primera mitad
csv_files = [
    #'common_player_info.csv', 'draft_combine_stats.csv', 'draft_history.csv', 
    'game.csv', 'game_info.csv','game_summary.csv', 'inactive_players.csv'
]

# Función para subir los archivos a Firebase
def upload_to_firebase(csv_list):
    missing = [name for name in csv_list if not os.path.isfile(os.path.join(csv_folder, name))]
    if missing:
        raise FileNotFoundError("Faltan CSV en PI_CSV_DIR: " + ", ".join(missing))
    for csv_file in csv_list:
        full_csv_path = os.path.join(csv_folder, csv_file)
        print(f"Subiendo archivo limpio: {csv_file}")

        # Cargar el CSV en un DataFrame
        df = pd.read_csv(full_csv_path)

        # Crear una colección en Firestore con el nombre del archivo CSV (sin extensión)
        collection_name = os.path.splitext(csv_file)[0]
        collection_ref = db.collection(collection_name)

        # Subir los datos a Firestore
        for index, row in df.iterrows():
            doc_ref = collection_ref.document(str(index))  # Usar el índice como ID del documento
            doc_ref.set(row.to_dict())  # Subir los datos como diccionario

        print(f"Datos limpios de {csv_file} subidos correctamente a Firestore.")

# Subir la primera mitad de los archivos
print("Subiendo la primera mitad de los archivos...")
upload_to_firebase(csv_files)

print("Primera mitad de los archivos CSV limpios subidos correctamente a Firebase Firestore.")


In [ ]:
# Archivos CSV para subir en la segunda mitad
csv_files_second_half = [
    'line_score.csv', 'officials.csv', 'other_stats.csv', 'player.csv', 'team.csv',
    'team_details.csv', 'team_history.csv', 'team_info_common.csv'
]

# Subir la segunda mitad de los archivos
print("Subiendo la segunda mitad de los archivos...")
upload_to_firebase(csv_files_second_half)

print("Segunda mitad de los archivos CSV limpios subidos correctamente a Firebase Firestore.")
